In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder

In [2]:
data = pd.read_csv("C:/projetos/GitHub/IA_Aulas/B.Engenharia_de_Atributos/credit_simple.csv", sep=";")
data.shape

(1000, 8)

In [3]:
data

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,CLASSE
0,1169.0,4,67,nenhum,01/01/2019,masculino solteiro,radio/tv,bom
1,5951.0,2,22,nenhum,01/01/2020,fem div/cas,radio/tv,ruim
2,2096.0,3,49,nenhum,02/01/2020,masculino solteiro,educação,bom
3,7882.0,4,45,nenhum,02/01/2019,masculino solteiro,mobilia/equipamento,bom
4,4870.0,4,53,nenhum,03/01/2018,masculino solteiro,carro novo,ruim
...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,29/06/2018,fem div/cas,mobilia/equipamento,bom
996,3857.0,4,40,nenhum,30/06/2018,masculino div/sep,carro usado,bom
997,804.0,4,38,nenhum,03/07/2018,masculino solteiro,radio/tv,bom
998,1845.0,4,23,nenhum,04/07/2019,masculino solteiro,radio/tv,ruim


In [4]:
X = data.drop(columns=["CLASSE"])
y = data["CLASSE"]

In [5]:
X.isnull().sum()

SALDO_ATUAL         7
RESIDENCIADESDE     0
IDADE               0
OUTROSPLANOSPGTO    0
DATA                0
ESTADOCIVIL         8
PROPOSITO           0
dtype: int64

In [6]:
#preenchendo valor com mediana
mediana = X["SALDO_ATUAL"].median()
X["SALDO_ATUAL"] = X["SALDO_ATUAL"].fillna(mediana)
X.isnull().sum()

SALDO_ATUAL         0
RESIDENCIADESDE     0
IDADE               0
OUTROSPLANOSPGTO    0
DATA                0
ESTADOCIVIL         8
PROPOSITO           0
dtype: int64

In [7]:
X.groupby(["ESTADOCIVIL"]).size()

ESTADOCIVIL
fem div/cas               308
masculino casado/viuvo     92
masculino div/sep          50
masculino solteiro        542
dtype: int64

In [8]:
# Sendo "masculino solteiro" a moda, entaão
X["ESTADOCIVIL"] = X["ESTADOCIVIL"].fillna("masculino solteiro")

In [9]:
print(X.groupby(["ESTADOCIVIL"]).size())
print(X.isnull().sum())

ESTADOCIVIL
fem div/cas               308
masculino casado/viuvo     92
masculino div/sep          50
masculino solteiro        550
dtype: int64
SALDO_ATUAL         0
RESIDENCIADESDE     0
IDADE               0
OUTROSPLANOSPGTO    0
DATA                0
ESTADOCIVIL         0
PROPOSITO           0
dtype: int64


In [10]:
#Usando desvio padrão para retirar Outliers
desv_padr = X["SALDO_ATUAL"].std()

X.loc[X["SALDO_ATUAL"] >= 2 * desv_padr, ["SALDO_ATUAL","PROPOSITO"]]

,SALDO_ATUAL,PROPOSITO
127,2.541111e+09,obras
160,2.154441e+10,radio/tv


In [11]:
X.loc[X["SALDO_ATUAL"] >= 2 * desv_padr, "SALDO_ATUAL"] = mediana #Mudando os outliers para valores da mediana

X.loc[X["SALDO_ATUAL"] >= 2 * desv_padr, ["SALDO_ATUAL","PROPOSITO"]]

,SALDO_ATUAL,PROPOSITO


In [12]:
#retirando Propositos pouco representados
print(X.groupby(["PROPOSITO"]).size())
X.loc[(X["PROPOSITO"]=="Eletrodomésticos") | (X["PROPOSITO"]=="qualificação"), 'PROPOSITO'] = 'outros'
print("------------------------")
print(X.groupby(["PROPOSITO"]).size())

PROPOSITO
Eletrodomésticos        12
carro novo             234
carro usado            103
educação                50
mobilia/equipamento    181
negócios                97
obras                   22
outros                  12
qualificação             9
radio/tv               280
dtype: int64
------------------------
PROPOSITO
carro novo             234
carro usado            103
educação                50
mobilia/equipamento    181
negócios                97
obras                   22
outros                  33
radio/tv               280
dtype: int64


In [13]:
X["DATA"] = pd.to_datetime(X["DATA"], format= '%d/%m/%Y')

In [14]:
X["DATA"]

0     2019-01-01
1     2020-01-01
2     2020-01-02
3     2019-01-02
4     2018-01-03
         ...    
995   2018-06-29
996   2018-06-30
997   2018-07-03
998   2019-07-04
999   2018-07-05
Name: DATA, Length: 1000, dtype: datetime64[us]

In [15]:
X["ANO"] = X["DATA"].dt.year
X["MES"] = X["DATA"].dt.month
X["DIA"] = X["DATA"].dt.day

X["DIA"]

0       1
1       1
2       2
3       2
4       3
       ..
995    29
996    30
997     3
998     4
999     5
Name: DIA, Length: 1000, dtype: int32

In [16]:
labelencoder = LabelEncoder()
X["ESTADOCIVIL"] = labelencoder.fit_transform(X["ESTADOCIVIL"])
X["PROPOSITO"] = labelencoder.fit_transform(X["PROPOSITO"])

In [17]:
X["OUTROSPLANOSPGTO"].unique()

<StringArray>
['nenhum', 'banco', 'stores']
Length: 3, dtype: str

>Fazendo One Hot Encoding

In [18]:

z = pd.get_dummies(X["OUTROSPLANOSPGTO"], prefix="OUTROS", dtype=int)
z

,OUTROS_banco,OUTROS_nenhum,OUTROS_stores
0,0,1,0
1,0,1,0
2,0,1,0
3,0,1,0
4,0,1,0
...,...,...,...
995,0,1,0
996,0,1,0
997,0,1,0
998,0,1,0


In [20]:
X = pd.concat([X, z], axis=1)


In [21]:
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIA,OUTROS_banco,OUTROS_nenhum,OUTROS_stores
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,1,0,1,0
1,5951.0,2,22,nenhum,2020-01-01,0,7,2020,1,1,0,1,0
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,2,0,1,0
3,7882.0,4,45,nenhum,2019-01-02,3,3,2019,1,2,0,1,0
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,3,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,2018-06-29,0,3,2018,6,29,0,1,0
996,3857.0,4,40,nenhum,2018-06-30,2,1,2018,6,30,0,1,0
997,804.0,4,38,nenhum,2018-07-03,3,7,2018,7,3,0,1,0
998,1845.0,4,23,nenhum,2019-07-04,3,7,2019,7,4,0,1,0


In [23]:
ss = StandardScaler()
X["SALDO_ATUAL"] = ss.fit_transform(X[["SALDO_ATUAL"]])
X["RESIDENCIADESDE"] = ss.fit_transform(X[["RESIDENCIADESDE"]])
X["IDADE"] = ss.fit_transform(X[["IDADE"]])
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIA,OUTROS_banco,OUTROS_nenhum,OUTROS_stores
0,-0.745516,1.046987,1.639276,nenhum,2019-01-01,3,7,2019,1,1,0,1,0
1,0.957740,-0.765977,-0.740241,nenhum,2020-01-01,0,7,2020,1,1,0,1,0
2,-0.415337,0.140505,0.687469,nenhum,2020-01-02,3,2,2020,1,2,0,1,0
3,1.645526,1.046987,0.475956,nenhum,2019-01-02,3,3,2019,1,2,0,1,0
4,0.572709,1.046987,0.898982,nenhum,2018-01-03,3,0,2018,1,3,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.543562,1.046987,-0.264338,nenhum,2018-06-29,0,3,2018,6,29,0,1,0
996,0.211898,1.046987,0.211566,nenhum,2018-06-30,2,1,2018,6,30,0,1,0
997,-0.875522,1.046987,0.105809,nenhum,2018-07-03,3,7,2018,7,3,0,1,0
998,-0.504738,1.046987,-0.687363,nenhum,2019-07-04,3,7,2019,7,4,0,1,0


In [26]:
X = X.drop(columns=["OUTROSPLANOSPGTO"])

In [27]:
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIA,OUTROS_banco,OUTROS_nenhum,OUTROS_stores
0,-0.745516,1.046987,1.639276,2019-01-01,3,7,2019,1,1,0,1,0
1,0.957740,-0.765977,-0.740241,2020-01-01,0,7,2020,1,1,0,1,0
2,-0.415337,0.140505,0.687469,2020-01-02,3,2,2020,1,2,0,1,0
3,1.645526,1.046987,0.475956,2019-01-02,3,3,2019,1,2,0,1,0
4,0.572709,1.046987,0.898982,2018-01-03,3,0,2018,1,3,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.543562,1.046987,-0.264338,2018-06-29,0,3,2018,6,29,0,1,0
996,0.211898,1.046987,0.211566,2018-06-30,2,1,2018,6,30,0,1,0
997,-0.875522,1.046987,0.105809,2018-07-03,3,7,2018,7,3,0,1,0
998,-0.504738,1.046987,-0.687363,2019-07-04,3,7,2019,7,4,0,1,0
